# LlamaIndex Tools

In [11]:
import os
from dotenv import load_dotenv

import chromadb

from llama_index.core import VectorStoreIndex
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.tools import FunctionTool, QueryEngineTool
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.tools.google import GmailToolSpec
from llama_index.tools.mcp import BasicMCPClient, McpToolSpec

In [12]:
load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")

In [13]:
def get_weather(location: str) -> str:
    """Useful for getting the weather for a given location."""
    print(f"Getting weather for {location}")
    return f"The weather in {location} is sunny."

tool = FunctionTool.from_defaults(
    get_weather,
    name="my_weather_tool",
    description="Useful for getting the weather for a given location.",
)

tool.call("Berlin")

Getting weather for Berlin


ToolOutput(blocks=[TextBlock(block_type='text', text='The weather in Berlin is sunny.')], tool_name='my_weather_tool', raw_input={'args': ('Berlin',), 'kwargs': {}}, raw_output='The weather in Berlin is sunny.', is_error=False)

Let's now create a query engine tool:

In [14]:
db = chromadb.PersistentClient(path="../alfred_chroma_db")
chroma_collection = db.get_or_create_collection("alfred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

In [15]:
embed_model = HuggingFaceEmbedding(
    model_name='BAAI/bge-small-en-v1.5',
)

llm = HuggingFaceInferenceAPI(
    model_name='meta-llama/Llama-3.3-70B-Instruct',
    temperature=0.7,
    max_tokens=100,
    token=HF_TOKEN,
    provider='auto',
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [6]:
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    embed_model=embed_model,
)

query_engine = index.as_query_engine(llm=llm)

tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name='query_engine_king',
    description='the best query engine tool'
)

In [7]:
await tool.acall(
    "Responds about research on the impact of AI on the future of work and society?"
)

ToolOutput(blocks=[TextBlock(block_type='text', text='There is no information available about research on the impact of AI on the future of work and society. The available information focuses on environmental conservation and sustainability, with one individual working on wetland ecosystems and climate change, and another on ecological conservation and sustainability in local government or a related organization. Neither of these areas mentions AI or its impact on work and society.')], tool_name='query_engine_king', raw_input={'input': 'Responds about research on the impact of AI on the future of work and society?'}, raw_output=Response(response='There is no information available about research on the impact of AI on the future of work and society. The available information focuses on environmental conservation and sustainability, with one individual working on wetland ecosystems and climate change, and another on ecological conservation and sustainability in local government or a rela

In [8]:
tool_spec = GmailToolSpec()
tool_spec_list = tool_spec.to_tool_list()
tool_spec_list

In [9]:
[print(tool.metadata.name, tool.metadata.description) for tool in tool_spec_list]

load_data load_data() -> List[llama_index.core.schema.Document]
Load emails from the user's account.
search_messages search_messages(query: str, max_results: Optional[int] = None)

        Searches email messages given a query string and the maximum number
        of results requested by the user
           Returns: List of relevant message objects up to the maximum number of results.

        Args:
            query (str): The user's query
            max_results (Optional[int]): The maximum number of search results
            to return.
create_draft create_draft(to: Optional[List[str]] = None, subject: Optional[str] = None, message: Optional[str] = None) -> str

        Create and insert a draft email.
           Print the returned draft's message and id.
           Returns: Draft object, including draft id and message meta data.

        Args:
            to (Optional[str]): The email addresses to send the message to
            subject (Optional[str]): The subject for the event
  

[None, None, None, None, None, None]

We can use tools associated with MCP:

In [10]:
mcp_client = BasicMCPClient("http://127.0.0.1:8000/sse")
mcp_tool = McpToolSpec(client=mcp_client)

# get the agent
agent = await get_agent(mcp_tool)

# create the agent context
agent_context = Context(agent)

NameError: name 'get_agent' is not defined